# Clients Cegelec NC : nettoyage de l'export Teepee « Entreprise »

Entrée : export Excel Teepee des entreprises (`TEEPEE_Entreprise_dataExport (3).xlsx`, non versionné car il contient des données clients).
Sortie : `df2_export.xlsx`, colonnes renommées, mails et contacts regroupés, adresses de facturation complétées, retours à la ligne activés pour *Mails* et *Contacts*.

In [ ]:
# installation des programmes
!pip install unidecode
!pip install selenium
!pip install ydata_profiling

In [ ]:
import requests
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib
import matplotlib.pyplot as plt
import json
from bs4 import BeautifulSoup
import re
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
import time
from selenium.webdriver import ChromeOptions
from datetime import datetime
from unidecode import unidecode
from ast import literal_eval
from itertools import chain
from ydata_profiling import ProfileReport
from openpyxl import load_workbook
from openpyxl.styles import Alignment, Font, PatternFill, Border, Side
from openpyxl.utils import get_column_letter

In [ ]:
#lire le fichier excel entreprises
df = pd.read_excel('/content/TEEPEE_Entreprise_dataExport (3).xlsx')

In [ ]:
#Bibliothèque de colonnes
colonnes_valide = {
    'Nom (Nom)': "Nom",
    'Identifiant entreprise (IdentifiantEntreprise)' : "Identifiant entreprise",
    'Matricule Client Codex (MatriculeClientCodex)' : "Matricule Client Codex",
    'Groupe (r_RelVersGroupe_1)' : "Groupe",
    'Mail signature BI (MailSignatureBI)' : "Mail signature BI",
    'Signature BI (SignatureBI)' : "Signature BI",
    'Mail signature BI  (MailSignatureBI_1)' : "MailSignatureBI_1",
    'Process de facturation (ProcessDeFacturation)' : "Process de facturation",
    "Type de contrat (TypeDeContrat)" : "Type de contrat",
    'Taux horaires Tech (TauxHorairesTech)' : "Taux horaires Tech",
    'Taux horaires RT (TauxHorairesRT)' : "Taux horaires RT",
    'Déplacements (DePlacements)' : "Déplacements",
    'Email (Email)' : "Email",
    'Email (Email_2)' : "Email_2",
    'Email (Email_1)': "Email_1",
    'Téléphone (TeLePhone)' : "Téléphone",
    'Web (Web)' : "Web",
    'Adresse (Adresse)': "Adresse",
    'Ville (Ville)' : "Ville",
    'Code  postal (CodePostal)' : "Code Postal",
    'Pays (Pays)' : "Pays",
    "(LAdresseDeLivraisonEstIdentiqueALAdresseDeFacturation) L'adresse de livraison est identique à l'adresse de facturation" : "L'adresse de livraison est identique à l'adresse de facturation",
    'Adresse (AdresseLivraison)' : "Adresse Livraison",
    'Ville (VilleLivraison)' : "Ville Livraison",
    'Code postal (CodePostalLivraison)' : "Code Postale Livraison",
    'Pays (PaysLivraison)' : "Pays Livraison",
    'Statut (Statut)' : "Statut",
    'Zone de déplacement (r_RelVersZoneDeDeplacement_1)' : "Zone de déplacement",
    ' (r_RelDevisVersSegmentationPayeurNiv1_1)' : "r_RelDevisVersSegmentationPayeurNiv1_1",
    ' (r_RelDevisVersSegmentationPayeurNiv2_1)' : "r_RelDevisVersSegmentationPayeurNiv2_1",
    ' (r_RelDevisVersSegmentationPayeurNiv3_1)' : "r_RelDevisVersSegmentationPayeurNiv3_1",
    ' (r_RelDevisVersSegmentationPayeurNiv4_1)' : "r_RelDevisVersSegmentationPayeurNiv4_1",
    ' (r_RelDevisVersSegmentationProcedeNiv1_1)' : "r_RelDevisVersSegmentationProcedeNiv1_1",
}

In [ ]:
# -- Suppression des colonnes inutiles ou trop vides / incohérentes
df = df.drop(columns=[
    "TEEPEE_ID (TEEPEE_ID)",
    " (r_RelDevisVersSegmentationProcedeNiv4_1)",
    " (r_RelDevisVersSegmentationProcedeNiv3_1)",
    " (r_RelDevisVersSegmentationProcedeNiv2_1)",
    "Porteur d'affaires principal (r_RelVersPorteursDaffaires_1)",
    "Matricule Client Codex (MatriculeClientCodex)",
    "Groupe (r_RelVersGroupe_1)",
    "Statut (Statut)"
      ])
# Supprime ligne vide et nom entreprise : "Autre"
df = df.dropna(subset=['Nom (Nom)'])
df.drop(df[df['Nom (Nom)'] == "AUTRE"].index, inplace=True)

# Renommer mes colonnes
df.rename(columns=colonnes_valide, inplace=True)

# -- Colonne unique de mails (\n) :

cols_mails = [ "Mail signature BI", "MailSignatureBI_1", "Email", "Email_2", "Email_1" ]

df["Mails"] = (
    df[cols_mails]
    .apply(
        lambda row: " ;\n".join(  # Changed separator to include a newline character
            row.dropna().astype(str).str.strip()
        )
        if row.dropna().size > 0 else np.nan,
        axis=1
    )
)

# -- Remplacer les 0 dans les colonnes par des np.nan
df['Taux horaires Tech'] = df['Taux horaires Tech'].replace(    0., np.nan)
df['Taux horaires RT'] = df['Taux horaires RT'].replace(     0., np.nan)
df['Déplacements'] = df['Déplacements'].replace(           0., np.nan)

# -- Contacts (\n)
df["Signature BI"] = df["Signature BI"].str.replace(r";|\t", "\n", regex=True)

df["Contacts"] = df.apply(
    lambda row: (
        f"{row['Signature BI']}\n{row['Téléphone']}"
        if pd.notna(row['Signature BI']) and pd.notna(row['Téléphone'])
        else (str(row['Signature BI']) if pd.notna(row['Signature BI']) else (str(row['Téléphone']) if pd.notna(row['Téléphone']) else np.nan))
    ),
    axis=1
)

# -- Adresses (Ville / Pays)
#Ville
Ville = []
for x in df["Ville Livraison"]:
  if x in ['Nouméa', 'NOUMEA','Nouméa Cedex','NOUMEA CEDEX','Noumea', 'NOUMEA ','NOUMEA Cedex','noumea','BP J5 98849 NOUMEA Cedex ',
          'Noumeaa']:
        Ville.append("NOUMEA")
  elif x in ["Koné"]:
    Ville.append("KONE")
  elif x in ['Mont-Dore', "PLUM / MONT DORE", "MONT DORE" ]:
    Ville.append("MONT-DORE")
  elif x in ["Pont des Français"]:
    Ville.append("PONT DES FRANÇAIS")
  else :
    Ville.append(x)
df["Ville Livraison"] = Ville

#Pays
Pays = []
for x in df["Pays Livraison"]:
  if x in ['Nouvelle-Calédonie', 'NOUVELLE CALEDONIE','NOUVELLE-CALEDONIE', 'NC', 'nc', 'Nouvelle Calédonie']:
    Pays.append("NOUVELLE-CALEDONIE")
  else :
    Pays.append(x)
df["Pays Livraison"] = Pays

# -- Adresses Facturation
mask = df["L'adresse de livraison est identique à l'adresse de facturation"] == "OUI"

df.loc[mask, "Adresse facturation"] = df.loc[mask, "Adresse Livraison"]
df.loc[mask, "CodePostal facturation"] = df.loc[mask, "Code Postale Livraison"]
df.loc[mask, "Ville facturation"] = df.loc[mask, "Ville Livraison"]
df.loc[mask, "Pays facturation"] = df.loc[mask, "Pays Livraison"]

# -- Info
df.info()

In [ ]:
df2 = df[['Nom',
          'Identifiant entreprise',
          "Mails",
          'Contacts',
          "Taux horaires Tech",
          'Taux horaires RT',
          'Déplacements',
          'Web',
          "Process de facturation",
          'Adresse facturation',
          'CodePostal facturation',
          'Ville facturation',
          'Pays facturation'
          ]]

In [ ]:
# telecharger le fichier avec des sauts de lignes dans la colonne mails
output_file = "df2_export.xlsx"

# Export classique
df2.to_excel(output_file, index=False)

# Ouvrir le fichier exporté
wb = load_workbook(output_file)
ws = wb.active

# Colonnes sur lesquelles activer le wrap text
cols_to_wrap = ["Mails", "Contacts"]

# Parcourt chaque nom de colonne ciblé
for col_name in cols_to_wrap:

    # Vérifier que la colonne existe dans le fichier (évite les erreurs)
    if col_name in [cell.value for cell in ws[1]]:
        col_index = [cell.value for cell in ws[1]].index(col_name) + 1

        # Activer le retour à la ligne sur toutes les cellules de la colonne
        for row in ws.iter_rows(min_col=col_index, max_col=col_index, min_row=2):
            for cell in row:
                cell.alignment = Alignment(wrap_text=True)

wb.save(output_file)

# Télécharger
from google.colab import files
files.download(output_file)